In [0]:
src_path = f"{REPO_ROOT}/src/quality/rules.py"
text = open(src_path).read()
print(text[-400:])
print("\nends with a return:", text.strip().endswith("return out"))

In [0]:
import os, sys, time, uuid, json
REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import yaml
from datetime import datetime, timezone
from pyspark.sql import functions as F, Row
from src.quality.rules import evaluate, verdict, failure_condition

CONFIG = f"{REPO_ROOT}/config/quality_rules.yaml"
with open(CONFIG) as fh:
    CONFIG_DOC = yaml.safe_load(fh)

print(f"{len(CONFIG_DOC)} tables, "
      f"{sum(len(t['rules']) for t in CONFIG_DOC)} rules")

In [0]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS transit.ops.quality_results (
    run_id STRING, run_at TIMESTAMP, table_name STRING, rule_name STRING,
    rule_type STRING, target STRING, severity STRING,
    rows_checked BIGINT, rows_failed BIGINT, failed_pct DOUBLE,
    passed BOOLEAN, error STRING, duration_s DOUBLE, params STRING)
""")

from src.quality.rules import ROW_LEVEL, evaluate_row_level_batch

def run_checks(config, overrides=None, write=True):
    """overrides: {table_name: DataFrame} to test against something other than the table."""
    overrides = overrides or {}
    run_id, results = str(uuid.uuid4())[:8], []
    now = lambda: datetime.now(timezone.utc)

    def resolve(name):
        return overrides.get(name, spark.table(name))

    def record(table, rule, checked, failed, err, secs):
        passed, pct = (False, 0.0) if err else verdict(checked, failed, rule)
        results.append(Row(run_id=run_id, run_at=now(), table_name=table,
            rule_name=rule["name"], rule_type=rule["type"],
            target=rule.get("column") or ",".join(rule.get("columns", [])),
            severity=rule.get("severity", "error"),
            rows_checked=int(checked), rows_failed=int(failed), failed_pct=float(pct),
            passed=bool(passed), error=err, duration_s=round(secs, 2),
            params=json.dumps(rule.get("params", {}))))

    for entry in config:
        table = entry["table"]
        try:
            df = resolve(table)
        except Exception as e:
            results.append(Row(run_id=run_id, run_at=now(), table_name=table,
                rule_name="table_exists", rule_type="table_exists", target="",
                severity="error", rows_checked=0, rows_failed=0, failed_pct=0.0,
                passed=False, error=str(e)[:200], duration_s=0.0, params="{}"))
            continue

        row_rules = [r for r in entry["rules"] if r["type"] in ROW_LEVEL]
        other     = [r for r in entry["rules"] if r["type"] not in ROW_LEVEL]

        if row_rules:
            t0 = time.time()
            try:
                batch = evaluate_row_level_batch(df, row_rules)
                secs = time.time() - t0
                for r in row_rules:
                    c, f_ = batch[r["name"]]
                    record(table, r, c, f_, None, secs / len(row_rules))
            except Exception as e:
                for r in row_rules:
                    record(table, r, 0, 0, str(e)[:200], 0.0)

        for rule in other:
            t0 = time.time()
            try:
                lookups = {}
                if rule["type"] == "referential_integrity":
                    lookups[rule["params"]["to"]] = resolve(rule["params"]["to"])
                c, f_ = evaluate(df, rule, lookups)
                record(table, rule, c, f_, None, time.time() - t0)
            except Exception as e:
                record(table, rule, 0, 0, str(e)[:200], time.time() - t0)

    out = spark.createDataFrame(results)
    if write:
        out.write.format("delta").mode("append").saveAsTable("transit.ops.quality_results")
    return run_id, out

print("runner ready")

In [0]:
t0 = time.time()
run_id, res = run_checks(CONFIG_DOC)
print(f"run {run_id} · {res.count()} rules · {time.time()-t0:.0f}s")

(res.groupBy("severity")
   .agg(F.sum(F.col("passed").cast("int")).alias("passed"),
        F.sum((~F.col("passed")).cast("int")).alias("failed"))
   .display())

res.filter("NOT passed").select(
    "table_name", "rule_name", "rule_type", "target", "severity",
    "rows_checked", "rows_failed", "failed_pct", "error").display()

In [0]:
stops = spark.table("transit.silver.stops")

broken = (stops
    .withColumn("stop_lat", F.when(F.col("stop_id") == stops.first()["stop_id"],
                                   F.lit(99.0)).otherwise(F.col("stop_lat")))
    .withColumn("stop_name", F.when(F.col("location_type") == 1, None)
                              .otherwise(F.col("stop_name")))
    .union(stops.limit(3)))                       # 3 duplicate stop_ids

stops_cfg = [c for c in CONFIG_DOC if c["table"] == "transit.silver.stops"]
_, bad_res = run_checks(stops_cfg, overrides={"transit.silver.stops": broken}, write=False)

bad_res.select("rule_name", "rows_checked", "rows_failed", "failed_pct", "passed").display()

failed = {r["rule_name"] for r in bad_res.collect() if not r["passed"]}
print("rules that fired:", sorted(failed))
assert {"stop_lat_in_region", "stop_name_not_null", "stop_id_unique"} <= failed, \
       "a deliberately corrupted input did not fail the right rules"
print("\nthe engine catches what it should")

In [0]:
errors = res.filter("severity = 'error' AND NOT passed")
n = errors.count()
if n:
    errors.select("table_name", "rule_name", "target", "rows_failed", "failed_pct", "error").show(truncate=False)
spark.table("transit.ops.quality_results").orderBy(F.desc("run_at")).limit(10).display()
assert n == 0, f"{n} error-severity rules failed"
print("all error-severity rules passed")